# Feature-set comparison

Same models and hyperparameters as Phase 1 (`configs/phase1_best_params.json`, tuned on B1's training window),
three input sets:

1. **Price history only** — own regular-price lags, rolling stats, change history, shelf/sale history.
   No calendar, no category, no external data.
2. **Price history + USD/TRY** — set 1 plus USD/TRY level and 1 / 7 / 30-day changes.
3. **All features** — all 50 features + COICOP / unit-basis categories.

B1 and B2 are development blocks; B3 is the holdout (reported, not used for decisions).

In [1]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":  # paths below are relative to the repository root
    os.chdir(Path.cwd().parent)

import json
import logging
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from src.evaluation import metrics as M
from src.models.classical_ml import phase1 as P
from src.preprocessing.features import feature_columns

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s: %(message)s",
                    handlers=[logging.FileHandler("logs/feature_sets.log"), logging.StreamHandler()], force=True)
log = logging.getLogger("feature_sets")
pd.set_option("display.width", 230)
pd.set_option("display.max_columns", 30)

PROCESSED = Path("data/processed")
feat = pd.read_parquet(PROCESSED / "gurmar_features.parquet")
BLOCKS = json.loads((PROCESSED / "gurmar_blocks.json").read_text())["blocks"]
best_params = json.loads(Path("configs/phase1_best_params.json").read_text())
MODELS = list(best_params)

PRICE_HISTORY = feature_columns(["lags", "rolling", "sale"])
SETS = {
    "1_price_history": (PRICE_HISTORY, False),
    "2_price_history+usd_try": (PRICE_HISTORY + feature_columns(["usd_try"]), False),
    "3_all_features": (feature_columns(), True),
}
for k, (cols, cat) in SETS.items():
    print(f"{k}: {len(cols)} features{' + categories' if cat else ''}")

1_price_history: 19 features
2_price_history+usd_try: 23 features
3_all_features: 50 features + categories


In [2]:
rows = []
for block in BLOCKS:
    test = P.split_rows(feat, block["test_start"], block["test_end"])
    for base_name, base_pred in M.baselines(test).items():
        rows.append(M.evaluate(test, base_pred, base_name).assign(block=block["block"], feature_set="—", algo=base_name))
    for set_name, (cols, categorical) in SETS.items():
        for name in MODELS:
            start = time.time()
            _, test_rows, pct, _ = P.train_block(name, best_params[name], feat, block, cols, categorical)
            ev = M.evaluate(test_rows, P.to_price(test_rows, pct), f"{set_name} | {name}")
            rows.append(ev.assign(block=block["block"], feature_set=set_name, algo=name))
            log.info("%s %s %s %.0fs", block["block"], set_name, name, time.time() - start)

cmp = M.add_improvement(pd.concat(rows, ignore_index=True))
cmp.to_csv(PROCESSED / "feature_set_comparison.csv", index=False)

2026-10-06 16:11:14,786 INFO feature_sets: B1 1_price_history lightgbm 16s


2026-10-06 16:11:30,500 INFO feature_sets: B1 1_price_history xgboost 16s


2026-10-06 16:11:33,424 INFO feature_sets: B1 1_price_history catboost 3s


2026-10-06 16:11:42,490 INFO feature_sets: B1 1_price_history lightgbm_hurdle 9s


2026-10-06 16:13:00,147 INFO feature_sets: B1 2_price_history+usd_try lightgbm 78s


...

2026-10-06 16:23:26,698 INFO feature_sets: B3 2_price_history+usd_try lightgbm_hurdle 98s


2026-10-06 16:24:47,187 INFO feature_sets: B3 3_all_features lightgbm 80s


2026-10-06 16:25:34,409 INFO feature_sets: B3 3_all_features xgboost 47s


2026-10-06 16:26:02,279 INFO feature_sets: B3 3_all_features catboost 28s


2026-10-06 16:28:37,757 INFO feature_sets: B3 3_all_features lightgbm_hurdle 155s


## Improvement over the better persistence baseline (%, positive = model better)

In [3]:
def table(blocks, subset):
    t = cmp[cmp["block"].isin(blocks) & (cmp["subset"] == subset) & (cmp["feature_set"] != "—")]
    return t.pivot_table(index=["feature_set", "algo"], columns="block", values="impr_vs_best_%").round(2)

table(["B1", "B2"], "spec target days")

block                                      B1     B2
feature_set             algo                        
1_price_history         catboost        -0.44  -0.26
                        lightgbm         4.46   8.36
                        lightgbm_hurdle  5.44  19.71
                        xgboost          3.35   0.90
2_price_history+usd_try catboost        -0.43  -2.28
                        lightgbm         4.46   7.28
                        lightgbm_hurdle  5.23  18.75
                        xgboost          3.16   1.23
3_all_features          catboost        -0.00  -6.40
                        lightgbm         4.24   6.69
                        lightgbm_hurdle  6.36  18.89
                        xgboost          2.98   0.43

In [4]:
table(["B1", "B2"], "data target days")

block                                      B1     B2
feature_set             algo                        
1_price_history         catboost        -0.47  -0.48
                        lightgbm         4.20  11.07
                        lightgbm_hurdle  5.24  23.84
                        xgboost          3.05   1.29
2_price_history+usd_try catboost        -0.39  -2.34
                        lightgbm         4.34   9.34
                        lightgbm_hurdle  5.16  23.68
                        xgboost          2.87   1.75
3_all_features          catboost         0.10  -8.47
                        lightgbm         3.92   8.32
                        lightgbm_hurdle  6.51  23.72
                        xgboost          2.61   0.66

In [5]:
table(["B1", "B2"], "global (model on spec days)")

block                                      B1    B2
feature_set             algo                       
1_price_history         catboost        -0.09 -0.04
                        lightgbm         0.91  1.22
                        lightgbm_hurdle  1.11  2.88
                        xgboost          0.68  0.13
2_price_history+usd_try catboost        -0.09 -0.33
                        lightgbm         0.91  1.06
                        lightgbm_hurdle  1.07  2.74
                        xgboost          0.65  0.18
3_all_features          catboost        -0.00 -0.94
                        lightgbm         0.87  0.98
                        lightgbm_hurdle  1.30  2.76
                        xgboost          0.61  0.06

In [6]:
table(["B1", "B2"], "all days (model everywhere)")

block                                      B1     B2
feature_set             algo                        
1_price_history         catboost        -1.11  -1.99
                        lightgbm         3.16   9.72
                        lightgbm_hurdle  5.17  18.03
                        xgboost          1.28   1.02
2_price_history+usd_try catboost        -3.00  -5.48
                        lightgbm         3.37   6.40
                        lightgbm_hurdle  5.29  18.37
                        xgboost          1.19   1.37
3_all_features          catboost        -1.37 -10.99
                        lightgbm         2.82   3.94
                        lightgbm_hurdle  6.79  16.68
                        xgboost          0.88   0.64

## MAE (TRY) — best algorithm per feature set, development blocks

In [7]:
dev = cmp[cmp["block"].isin(["B1", "B2"])]
summary = []
for subset in ["spec target days", "data target days", "global (model on spec days)", "all days (model everywhere)"]:
    part = dev[dev["subset"] == subset]
    base = part[part["algo"] == "persistence_causal_regular"].groupby("block")["MAE"].first()
    for fs in SETS:
        sub = part[part["feature_set"] == fs]
        best_algo = sub.groupby("algo")["impr_vs_best_%"].mean().idxmax()
        b = sub[sub["algo"] == best_algo].set_index("block")
        summary.append({"subset": subset, "feature_set": fs, "best_algo": best_algo,
                        "MAE_B1": b.loc["B1", "MAE"], "MAE_B2": b.loc["B2", "MAE"],
                        "persistence_B1": base["B1"], "persistence_B2": base["B2"],
                        "impr_B1_%": b.loc["B1", "impr_vs_best_%"], "impr_B2_%": b.loc["B2", "impr_vs_best_%"],
                        "DA_B1_%": b.loc["B1", "DA_%"], "DA_B2_%": b.loc["B2", "DA_%"]})
pd.DataFrame(summary).round(3)

,subset,feature_set,best_algo,MAE_B1,MAE_B2,persistence_B1,persistence_B2,impr_B1_%,impr_B2_%,DA_B1_%,DA_B2_%
0,spec target days,1_price_history,lightgbm_hurdle,3.522,4.050,3.725,5.045,5.445,19.711,92.996,92.290
1,spec target days,2_price_history+usd_try,lightgbm_hurdle,3.531,4.099,3.725,5.045,5.227,18.748,93.024,92.192
2,spec target days,3_all_features,lightgbm_hurdle,3.488,4.092,3.725,5.045,6.360,18.891,93.081,92.400
3,data target days,1_price_history,lightgbm_hurdle,2.534,3.785,2.675,4.970,5.242,23.837,94.852,94.254
4,data target days,2_price_history+usd_try,lightgbm_hurdle,2.537,3.793,2.675,4.970,5.164,23.684,94.910,94.207
5,data target days,3_all_features,lightgbm_hurdle,2.500,3.791,2.675,4.970,6.513,23.720,94.981,94.323
6,global (model on spec days),1_price_history,lightgbm_hurdle,1.814,3.861,1.835,3.976,1.113,2.882,96.084,95.177
7,global (model on spec days),2_price_history+usd_try,lightgbm_hurdle,1.815,3.867,1.835,3.976,1.068,2.741,96.087,95.166
8,global (model on spec days),3_all_features,lightgbm_hurdle,1.811,3.866,1.835,3.976,1.300,2.762,96.093,95.190
9,all days (model everywhere),1_price_history,lightgbm_hurdle,1.740,3.259,1.835,3.976,5.171,18.026,96.461,96.344


## Holdout B3 (report only)

In [8]:
table(["B3"], "spec target days").join(table(["B3"], "global (model on spec days)"), rsuffix=" global")

block                                        B3  B3 global
feature_set             algo                              
1_price_history         catboost         -31.99      -2.34
                        lightgbm         -50.53      -3.70
                        lightgbm_hurdle    6.61       0.48
                        xgboost          -57.48      -4.20
2_price_history+usd_try catboost         -18.68      -1.37
                        lightgbm         -48.09      -3.52
                        lightgbm_hurdle   14.82       1.08
                        xgboost          -65.07      -4.76
3_all_features          catboost          -3.74      -0.27
                        lightgbm         -82.34      -6.02
                        lightgbm_hurdle  -50.80      -3.71
                        xgboost         -111.70      -8.17